# University Timetable GA

Genetic algorithm timetable optimization. Edit courses, faculty, timeslots, and GA settings below. The chromosome assigns one timeslot per course; the fitness penalizes faculty collisions and imbalanced daily loads.

Run the code cell to generate **`university-timetable-ga.png`** in the repository folder.

In [ ]:
# Editable inputs
COURSES=["CS101","CS102","CS103","EE101","EE102","MATH101"]
FACULTY=["Sharma","Patel","Sharma","Iyer","Iyer","Nair"]
TIMESLOTS=["Mon 9","Mon 11","Wed 9","Wed 11","Fri 9"]
POPULATION, GENERATIONS, MUTATION, SEED=100,250,0.12,42
OUTPUT="university-timetable-ga.png"
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(SEED)
import random
random.seed(SEED)
if not COURSES or len(COURSES)!=len(FACULTY) or not TIMESLOTS: raise ValueError("Courses and faculty must have equal nonzero length")
days=[s.split()[0] for s in TIMESLOTS]; ds=set(days); target=len(COURSES)/len(ds)
def cost(x):
 c=sum(x[i]==x[j] and FACULTY[i]==FACULTY[j] for i in range(len(x)) for j in range(i)); loads=[sum(days[s]==d for s in x) for d in ds]; return 1000*c+sum((n-target)**2 for n in loads)
pop=[[random.randrange(len(TIMESLOTS)) for _ in COURSES] for _ in range(POPULATION)]; hist=[]
for _ in range(GENERATIONS):
 pop.sort(key=cost); hist.append(cost(pop[0])); nxt=[pop[0][:]]
 while len(nxt)<POPULATION:
  a,b=[random.choice(pop[:max(2,POPULATION//3)])[:] for _ in range(2)]; cut=random.randrange(len(COURSES)+1); child=a[:cut]+b[cut:]
  for i in range(len(child)):
   if random.random()<MUTATION: child[i]=random.randrange(len(TIMESLOTS))
  nxt.append(child)
 pop=nxt
best=min(pop,key=cost); hist.append(cost(best)); print("Best penalty:",cost(best))
for c,f,s in zip(COURSES,FACULTY,best): print(f"{c:10} {f:10} {TIMESLOTS[s]}")
fig,(a,b)=plt.subplots(2,1,figsize=(12,9)); a.scatter(best,range(len(best)),s=650,c=range(len(best)),cmap="Set2");
for i,(c,f) in enumerate(zip(COURSES,FACULTY)): a.text(best[i],i,f"{c}\n{f}",ha="center",va="center",fontsize=8)
a.set(xticks=range(len(TIMESLOTS)),xticklabels=TIMESLOTS,yticks=[],title="Best timetable"); a.set_ylim(len(COURSES)-.35,-.65); a.grid(axis="x",alpha=.25); b.plot(hist); b.set(title="Best penalty by generation",xlabel="Generation",ylabel="Penalty"); b.grid(alpha=.2); fig.tight_layout(); fig.savefig(OUTPUT,dpi=160); plt.show(); print("Saved",OUTPUT)